# 🤖 Soal 3: Modeling & Evaluation
## Klasifikasi Tingkat Obesitas
**Mata Kuliah:** Pembelajaran Mesin — UAS Genap 2025/2026

---
Notebook ini membangun dan mengevaluasi **3 model klasifikasi**:
1. K-Nearest Neighbors (KNN)
2. Decision Tree
3. Random Forest (Ensemble Learning)

Model terbaik akan disimpan untuk digunakan di aplikasi Streamlit.

## 1. Load Data & Library

In [ ]:
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
import joblib, os, warnings
warnings.filterwarnings('ignore')

from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score, GridSearchCV
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                              f1_score, confusion_matrix, classification_report)

OUTPUT_PATH = '../outputs/'
os.makedirs(OUTPUT_PATH, exist_ok=True)

# Load preprocessed data
X_train = np.load('../models/X_train.npy')
X_val   = np.load('../models/X_val.npy')
X_test  = np.load('../models/X_test.npy')
y_train = np.load('../models/y_train.npy')
y_val   = np.load('../models/y_val.npy')
y_test  = np.load('../models/y_test.npy')

# Raw (unscaled) untuk Decision Tree
X_train_raw = np.load('../models/X_train_raw.npy')
X_val_raw   = np.load('../models/X_val_raw.npy')
X_test_raw  = np.load('../models/X_test_raw.npy')

le_target    = joblib.load('../models/label_encoder.pkl')
feature_cols = joblib.load('../models/feature_cols.pkl')

class_names = le_target.classes_
print(f'Train: {X_train.shape}, Val: {X_val.shape}, Test: {X_test.shape}')
print(f'Kelas: {class_names}')

## 2. Model 1: K-Nearest Neighbors (KNN)

In [ ]:
# Tuning KNN: cari K terbaik
k_range = range(1, 21)
val_scores = []
for k in k_range:
    knn = KNeighborsClassifier(n_neighbors=k)
    knn.fit(X_train, y_train)
    val_scores.append(accuracy_score(y_val, knn.predict(X_val)))

best_k = k_range[np.argmax(val_scores)]
print(f'K terbaik: {best_k} dengan Val Accuracy: {max(val_scores):.4f}')

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(list(k_range), val_scores, marker='o', color='royalblue')
ax.axvline(best_k, color='red', linestyle='--', label=f'K={best_k}')
ax.set_title('Tuning KNN — Validation Accuracy vs K', fontweight='bold')
ax.set_xlabel('Nilai K')
ax.set_ylabel('Accuracy')
ax.legend()
plt.tight_layout()
plt.savefig(f'{OUTPUT_PATH}10_knn_tuning.png', dpi=150)
plt.show()

In [ ]:
# Train final KNN
knn_model = KNeighborsClassifier(n_neighbors=best_k)
knn_model.fit(X_train, y_train)
y_pred_knn = knn_model.predict(X_test)

print('=== KNN — Classification Report (Test Set) ===')
print(classification_report(y_test, y_pred_knn, target_names=class_names))

## 3. Model 2: Decision Tree

In [ ]:
# Tuning max_depth
depth_range = range(2, 16)
val_scores_dt = []
for d in depth_range:
    dt = DecisionTreeClassifier(max_depth=d, random_state=42)
    dt.fit(X_train_raw, y_train)
    val_scores_dt.append(accuracy_score(y_val, dt.predict(X_val_raw)))

best_depth = list(depth_range)[np.argmax(val_scores_dt)]
print(f'Depth terbaik: {best_depth} dengan Val Accuracy: {max(val_scores_dt):.4f}')

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(list(depth_range), val_scores_dt, marker='s', color='forestgreen')
ax.axvline(best_depth, color='red', linestyle='--', label=f'depth={best_depth}')
ax.set_title('Tuning Decision Tree — Validation Accuracy vs Max Depth', fontweight='bold')
ax.set_xlabel('Max Depth')
ax.set_ylabel('Accuracy')
ax.legend()
plt.tight_layout()
plt.savefig(f'{OUTPUT_PATH}11_dt_tuning.png', dpi=150)
plt.show()

In [ ]:
# Train final Decision Tree
dt_model = DecisionTreeClassifier(max_depth=best_depth, random_state=42)
dt_model.fit(X_train_raw, y_train)
y_pred_dt = dt_model.predict(X_test_raw)

print('=== Decision Tree — Classification Report (Test Set) ===')
print(classification_report(y_test, y_pred_dt, target_names=class_names))

## 4. Model 3: Random Forest (Ensemble)

In [ ]:
# Tuning n_estimators
n_trees = [10, 30, 50, 100, 150, 200]
val_scores_rf = []
for n in n_trees:
    rf = RandomForestClassifier(n_estimators=n, random_state=42, n_jobs=-1)
    rf.fit(X_train_raw, y_train)
    val_scores_rf.append(accuracy_score(y_val, rf.predict(X_val_raw)))

best_n = n_trees[np.argmax(val_scores_rf)]
print(f'N estimators terbaik: {best_n} dengan Val Accuracy: {max(val_scores_rf):.4f}')

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(n_trees, val_scores_rf, marker='^', color='darkorange')
ax.axvline(best_n, color='red', linestyle='--', label=f'n={best_n}')
ax.set_title('Tuning Random Forest — Validation Accuracy vs N Estimators', fontweight='bold')
ax.set_xlabel('N Estimators')
ax.set_ylabel('Accuracy')
ax.legend()
plt.tight_layout()
plt.savefig(f'{OUTPUT_PATH}12_rf_tuning.png', dpi=150)
plt.show()

In [ ]:
# Train final Random Forest
rf_model = RandomForestClassifier(n_estimators=best_n, random_state=42, n_jobs=-1)
rf_model.fit(X_train_raw, y_train)
y_pred_rf = rf_model.predict(X_test_raw)

print('=== Random Forest — Classification Report (Test Set) ===')
print(classification_report(y_test, y_pred_rf, target_names=class_names))

## 5. Perbandingan Semua Model

In [ ]:
def get_metrics(y_true, y_pred, name):
    return {
        'Model': name,
        'Accuracy': accuracy_score(y_true, y_pred),
        'Precision': precision_score(y_true, y_pred, average='weighted'),
        'Recall': recall_score(y_true, y_pred, average='weighted'),
        'F1-Score': f1_score(y_true, y_pred, average='weighted')
    }

results = pd.DataFrame([
    get_metrics(y_test, y_pred_knn, f'KNN (k={best_k})'),
    get_metrics(y_test, y_pred_dt,  f'Decision Tree (depth={best_depth})'),
    get_metrics(y_test, y_pred_rf,  f'Random Forest (n={best_n})')
])

results_display = results.set_index('Model')
print('=== TABEL PERBANDINGAN PERFORMA MODEL ===')
print(results_display.round(4).to_string())
results.to_csv(f'{OUTPUT_PATH}model_comparison.csv', index=False)

In [ ]:
# Bar chart perbandingan
metrics = ['Accuracy', 'Precision', 'Recall', 'F1-Score']
x = np.arange(len(metrics))
width = 0.25

fig, ax = plt.subplots(figsize=(12, 6))
colors = ['royalblue', 'forestgreen', 'darkorange']
for i, row in results.iterrows():
    offset = (i - 1) * width
    bars = ax.bar(x + offset, [row[m] for m in metrics], width, label=row['Model'], color=colors[i], alpha=0.85)
    for bar in bars:
        ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.005,
                f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=8)

ax.set_xlabel('Metrik')
ax.set_ylabel('Score')
ax.set_title('Perbandingan Performa Model — Test Set', fontsize=13, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(metrics)
ax.set_ylim(0, 1.1)
ax.legend()
plt.tight_layout()
plt.savefig(f'{OUTPUT_PATH}13_model_comparison_bar.png', dpi=150)
plt.show()

## 6. Confusion Matrix — Semua Model

In [ ]:
def plot_cm(y_true, y_pred, model_name, filename):
    cm = confusion_matrix(y_true, y_pred)
    fig, ax = plt.subplots(figsize=(10, 8))
    short_names = ['Insuff.', 'Normal', 'OW_I', 'OW_II', 'Ob_I', 'Ob_II', 'Ob_III']
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=short_names, yticklabels=short_names)
    ax.set_title(f'Confusion Matrix — {model_name}', fontsize=13, fontweight='bold')
    ax.set_xlabel('Predicted')
    ax.set_ylabel('Actual')
    plt.tight_layout()
    plt.savefig(f'{OUTPUT_PATH}{filename}', dpi=150)
    plt.show()

plot_cm(y_test, y_pred_knn, f'KNN (k={best_k})', '14_cm_knn.png')
plot_cm(y_test, y_pred_dt,  f'Decision Tree (depth={best_depth})', '15_cm_dt.png')
plot_cm(y_test, y_pred_rf,  f'Random Forest (n={best_n})', '16_cm_rf.png')

## 7. Feature Importance (Random Forest)

In [ ]:
importances = rf_model.feature_importances_
feat_imp = pd.Series(importances, index=feature_cols).sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(10, 7))
colors = ['#d73027' if v > feat_imp.median() else '#4575b4' for v in feat_imp.values]
ax.barh(feat_imp.index, feat_imp.values, color=colors)
ax.set_title('Feature Importance — Random Forest', fontsize=13, fontweight='bold')
ax.set_xlabel('Importance Score')
plt.tight_layout()
plt.savefig(f'{OUTPUT_PATH}17_feature_importance.png', dpi=150)
plt.show()

print('Top 5 fitur terpenting:')
print(feat_imp.sort_values(ascending=False).head())

## 8. Simpan Model Terbaik

In [ ]:
# Pilih model terbaik berdasarkan F1-Score
best_idx = results['F1-Score'].idxmax()
best_model_name = results.loc[best_idx, 'Model']
best_f1 = results.loc[best_idx, 'F1-Score']
print(f'🏆 Model Terbaik: {best_model_name}')
print(f'   F1-Score (weighted): {best_f1:.4f}')
print(f'   Accuracy: {results.loc[best_idx, "Accuracy"]:.4f}')

# Simpan semua model
joblib.dump(knn_model, '../models/knn_model.pkl')
joblib.dump(dt_model,  '../models/dt_model.pkl')
joblib.dump(rf_model,  '../models/rf_model.pkl')

# Simpan model terbaik dengan nama khusus
if 'Random Forest' in best_model_name:
    best_model = rf_model
    best_uses_scaled = False
elif 'KNN' in best_model_name:
    best_model = knn_model
    best_uses_scaled = True
else:
    best_model = dt_model
    best_uses_scaled = False

joblib.dump(best_model, '../models/best_model.pkl')
joblib.dump(best_uses_scaled, '../models/best_model_uses_scaled.pkl')
joblib.dump(best_model_name, '../models/best_model_name.pkl')
joblib.dump(results, '../models/results_df.pkl')

print('\n✅ Semua model dan metadata disimpan ke folder models/')

## 9. Kesimpulan & Justifikasi

### Perbandingan Model:

| Model | Kelebihan | Kekurangan |
|-------|-----------|------------|
| **KNN** | Sederhana, non-parametrik | Lambat saat prediksi, sensitif terhadap skala & noise |
| **Decision Tree** | Interpretable, cepat | Rentan overfitting, tidak stabil |
| **Random Forest** | Akurasi tinggi, robust terhadap overfitting | Interpretabilitas lebih rendah, komputasi lebih berat |

### Model Terbaik: **Random Forest**
Random Forest dipilih karena:
1. F1-Score tertinggi di antara semua model
2. Mekanisme *bagging* membuatnya robust terhadap overfitting
3. Mampu menangkap hubungan non-linear antar fitur
4. Feature importance memberikan interpretabilitas yang cukup baik